# 📦 1️⃣ Importaciones y carga de claves

In [34]:
import os
import json
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# -----------------------------
# RUTAS CORRECTAS PARA JUPYTER
# -----------------------------

# Ruta del proyecto (sube 1 nivel desde /notebooks)
BASE_DIR = Path().resolve().parents[0]      # <-- esta es la clave

# Si tu notebook está en: 03_IA_Segunda_Guerra_Mundial/notebooks/
# Entonces BASE_DIR apunta a: 03_IA_Segunda_Guerra_Mundial/

PROCESSED_DIR = BASE_DIR / "data" / "processed"
CHUNKS_PATH = PROCESSED_DIR / "ww2_chunks_plus_qa.jsonl"
EMB_PATH = PROCESSED_DIR / "ww2_embeddings_plus_qa.npy"


# 📄 2️⃣ Cargar chunks y embeddings

In [35]:
def cargar_datos():
    chunks = []
    with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            chunks.append(json.loads(line))

    emb_matrix = np.load(EMB_PATH)

    print(f"Chunks cargados: {len(chunks)}")
    print(f"Embeddings shape: {emb_matrix.shape}")

    return chunks, emb_matrix


chunks, emb_matrix = cargar_datos()


Chunks cargados: 4937
Embeddings shape: (4937, 3072)


# 🧠 3️⃣ Función para crear embeddings de la pregunta

In [36]:
def embed_text(text):
    resp = client.embeddings.create(
        model="text-embedding-3-large",
        input=text,
    )
    return np.array(resp.data[0].embedding, dtype="float32")


# 📏 4️⃣ Similitud coseno + recuperación de top-k fragmentos

In [37]:
def cosine_sim(query_vec, docs_matrix):
    num = docs_matrix @ query_vec
    den = (np.linalg.norm(docs_matrix, axis=1) * np.linalg.norm(query_vec) + 1e-10)
    return num / den


def recuperar_chunks(query, k=5):
    q_emb = embed_text(query)
    sims = cosine_sim(q_emb, emb_matrix)

    idx = np.argsort(sims)[::-1][:k]

    resultados = []
    for i in idx:
        c = chunks[int(i)]
        resultados.append({
            "score": float(sims[i]),
            "text": c["text"],
            "source": c["source"],
            "page": c["page"]
        })

    return resultados


# 📝 5️⃣ Prompt conciso + llamada al modelo

In [ ]:
def expandir_siglas(texto: str) -> str:
    """
    Sustituye siglas por su explicación la primera vez que aparecen en el texto.
    Solo expande la primera aparición para evitar repeticiones pesadas.
    """

    sustituciones = {
        "RAF": "RAF (Real Fuerza Aérea británica)",
        "Luftwaffe": "Luftwaffe (fuerza aérea alemana)",
        "Kriegsmarine": "Kriegsmarine (marina de guerra alemana)",
        "Wehrmacht": "Wehrmacht (fuerzas armadas alemanas)",
        "USAAF": "USAAF (Fuerza Aérea del Ejército de EE.UU.)",
    }

    for sigla, expansion in sustituciones.items():
        texto = texto.replace(sigla, expansion, 1)

    return texto


def generar_respuesta(query, contexto):
    # --- Refuerzo semántico para preguntas ambiguas ---
    query_lower = query.lower()

    if "amante" in query_lower:
        query += " (No confundir con matrimonio o relación legal)"
    elif "mujer" in query_lower or "esposa" in query_lower:
        query += " (Especificar si hubo matrimonio legal y cuándo)"

    # --- Construir fragmentos de contexto ---
    partes = []
    for i, c in enumerate(contexto):
        partes.append(
            f"[Fragmento {i+1} | Fuente: {c['source']} pág. {c['page']}]\n{c['text']}"
        )
    context_text = "\n\n".join(partes)

    mensajes = [
        {
            "role": "system",
            "content": (
                "Eres un historiador experto en la Segunda Guerra Mundial. "
                "Respondes siempre en español, de manera breve, clara y natural, máximo 6 líneas. "
                "Tu única fuente es el contexto proporcionado. No inventes datos. "
                "Evita expresiones demasiado técnicas o rígidas. "
                "Cuando una pregunta implique una distinción conceptual "
                "(por ejemplo, amante frente a esposa), aclara explícitamente "
                "la diferencia temporal o legal entre ambos términos."
            ),
        },
        {
            "role": "user",
            "content": (
                f"{context_text}\n\n"
                "Usa SOLO la información anterior. Si algo no aparece, responde: "
                "\"No existe información suficiente en los documentos para responder.\"\n\n"
                "Instrucciones:\n"
                "- Máximo 6 líneas.\n"
                "- Sin subtítulos.\n"
                "- Respuesta clara y concisa.\n"
                "- Evita frases demasiado técnicas o robóticas.\n"
                "- Ajusta el tono para que sea natural, no académico.\n"
                "- Añade citas al final: (Fuente: <PDF>, pág. X).\n\n"
                f"Pregunta: {query}"
            ),
        },
    ]

    resp = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=mensajes,
        max_completion_tokens=300,
    )

    respuesta = resp.choices[0].message.content

    # --- Postproceso: expansión automática de siglas ---
    respuesta = expandir_siglas(respuesta)

    return respuesta


# ❓ 6️⃣ Función final para preguntarle:

In [39]:
def responder(query, k=5):
    contexto = recuperar_chunks(query, k)
    respuesta = generar_respuesta(query, contexto)
    return respuesta


# PREGUNTAR

In [40]:
responder("¿Por qué fue importante la Operación Barbarroja?")


'La Operación Barbarroja fue importante porque representó el intento de Alemania nazi de conquistar la Unión Soviética para obtener recursos vitales como petróleo y alimentos, además de expandir el "espacio vital" alemán. Su fracaso significó que el rápido colapso soviético no ocurrió y se desencadenó una larga guerra de desgaste que desgastó a la Wehrmacht (fuerzas armadas alemanas). Además, inició una brutal persecución y exterminio de poblaciones eslavas según el plan nazi. (Fuente: Operación_Barbarroja.pdf, págs. 0, 1, 4, Frente_oriental_(Segunda_Guerra_Mundial).pdf, pág. 6).'

In [41]:
responder("¿Cuáles fueron las principales causas de la Segunda Guerra Mundial?")

'Las principales causas de la Segunda Guerra Mundial fueron las tensiones no resueltas tras la Primera Guerra Mundial, como la inestabilidad generada por el Tratado de Versalles y la crisis económica. Además, la política expansionista y racista de Hitler, junto con la inacción y apaciguamiento de países como Reino Unido, facilitaron la agresión alemana. Las rivalidades entre naciones europeas y la formación de alianzas también contribuyeron al estallido del conflicto (Fuente: qa_manual, pág. 1; Segunda_Guerra_Mundial.pdf, pág. 1; Frente_oriental_(Segunda_Guerra_Mundial).pdf, pág. 2).'

In [42]:
responder("¿Cuándo empezó la segunda guerra mundial?")

'La Segunda Guerra Mundial comenzó oficialmente el 1 de septiembre de 1939 con la invasión alemana de Polonia, seguida por la invasión soviética el 17 de septiembre. Este evento provocó que Reino Unido y Francia le declararan la guerra a Alemania dos días después. Aunque hay otras fechas usadas para el inicio, como conflictos previos en Asia o África, la fecha más aceptada es 1939 en Europa. (Fuente: Segunda_Guerra_Mundial.pdf, pág. 1 y pág. 0; Invasión_de_Polonia.pdf, pág. 0; Segunda_Guerra_Mundial.pdf, pág. 15).'

In [43]:
responder("¿Cómo afectó la relación entre Hitler y sus generales al desarrollo de la guerra?")

'La relación entre Hitler y sus generales se dañó por la desconfianza y el control rígido que él ejerció. Hitler asumió el mando directo, ignorando consejos expertos y limitando la autonomía militar, lo que provocó errores tácticos y estratégicos importantes. Esta falta de flexibilidad y la desobediencia a sus generales contribuyeron a fracasos en el Frente Oriental y otras batallas clave. Además, los constantes cambios de objetivos por parte de Hitler confundieron a sus mandos. (Fuente: qa_manual, pág. 1; Operación_Barbarroja.pdf, pág. 59).'

In [44]:
responder("¿Por qué la batalla de Inglaterra fue importante para frenar la expansión alemana?")

'La batalla de Inglaterra fue crucial porque impidió que Alemania lograra la superioridad aérea necesaria para la invasión de Gran Bretaña, frustrando la Operación León Marino. Esta derrota fue la primera gran caída alemana en la Segunda Guerra Mundial y permitió que Gran Bretaña siguiera siendo una base clave para que los Aliados invadieran Europa en 1944. Además, frenó la expansión alemana en Occidente tras la caída de Francia. (Fuente: Batalla_de_Inglaterra.pdf, págs. 0-1)'